# 06 — Latent market-state discovery around q99

## Research question

Do the observable five-minute market configurations discovered in Modules 03–05 organize themselves into recurrent latent states, and do any **filtered** states or state transitions carry elevated future q99 volatility hazard?

Module 06 introduces the first latent-state model in the Volatility Dynamics branch.

## What is deliberately *not* assumed

- no state is called Mean Reversion, Momentum, VOL, Bridge, Clean or Noisy in advance;
- no duration distribution is imposed beyond the standard HMM geometric implication;
- q99 is not an emission feature and is not used to fit the HMM;
- `return_5m` is used only to construct the external q99 evaluation target;
- no HSMM, transition covariates, shock-bypass gate, or trading layer is introduced yet.

The HMM must first earn the right to exist as a useful statistical representation.


In [ ]:
from pathlib import Path
import json, os, sys, warnings
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 320)
pd.set_option('display.max_rows', 300)

load_dotenv()
DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
DB_PATH = DATA_ROOT / 'database' / 'research.duckdb'
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    raise RuntimeError('Repository root not found.')
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_engineering.modeling import volatility_hmm_states as vh

START = os.getenv('VD6_START', '').strip() or None
END = os.getenv('VD6_END', '').strip() or None
ASSET_FILTER = [x.strip().upper() for x in os.getenv('VD6_ASSETS', '').split(',') if x.strip()]
TRAIN_FRACTION = float(os.getenv('VD6_TRAIN_FRACTION', str(vh.DEFAULT_TRAIN_FRACTION)))
CORE_K = tuple(int(x) for x in os.getenv('VD6_CORE_K', '2,3,4,5').split(',') if x.strip())
L2_K = tuple(int(x) for x in os.getenv('VD6_L2_K', '2,3,4').split(',') if x.strip())
N_INIT_CORE = int(os.getenv('VD6_N_INIT_CORE', str(vh.DEFAULT_N_INIT_CORE)))
N_INIT_L2 = int(os.getenv('VD6_N_INIT_L2', str(vh.DEFAULT_N_INIT_L2)))
MAX_ITER_CORE = int(os.getenv('VD6_MAX_ITER_CORE', str(vh.DEFAULT_MAX_ITER_CORE)))
MAX_ITER_L2 = int(os.getenv('VD6_MAX_ITER_L2', str(vh.DEFAULT_MAX_ITER_L2)))
RUN_L2 = os.getenv('VD6_RUN_L2', '1').strip().lower() not in {'0','false','no'}
OUT = PROJECT_ROOT / 'notebooks' / 'Volatility_Dynamics' / '06'
OUT.mkdir(parents=True, exist_ok=True)

print('Module 06 engine:', vh.VERSION)
print('selftest:', vh.selftest())
print('core K:', CORE_K, '| L2 K:', L2_K, '| RUN_L2:', RUN_L2)


## 1. Frozen emission feature contract

The core HMM uses the broad precursor families that had substantial history in Modules 03–05:

\[
X_t=[Range,Volume,TradeCount,ATMIV,Put25IV,Call25IV,Spread,BookUpdates].
\]

The sparse L2 variables are **not** allowed to shorten the main history. They are tested in a separate enhanced model.


In [ ]:
con = duckdb.connect(str(DB_PATH), read_only=True)
schema = con.execute('DESCRIBE time_series_feature_state').df()
available = set(schema['column_name'].astype(str))
contract = vh.feature_contract(available)
display(contract)
core_features = [f for f in vh.CORE_FEATURES if f in available]
l2_features = [f for f in vh.L2_FEATURES if f in available]
if len(core_features) != len(vh.CORE_FEATURES):
    raise RuntimeError(f'Missing core features: {set(vh.CORE_FEATURES)-set(core_features)}')
enhanced_features = core_features + l2_features


## 2. Load the synchronized five-minute panel

`return_5m` is loaded solely for q99 target construction. It is never placed in the HMM emission matrix.


In [ ]:
def qident(name):
    return '"' + str(name).replace('"', '""') + '"'

select_cols = list(dict.fromkeys(['state_time','underlying','return_5m'] + enhanced_features))
where = []
if START:
    ts = pd.Timestamp(START)
    ts = ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')
    where.append(f"state_time >= TIMESTAMPTZ '{ts.isoformat()}'")
if END:
    ts = pd.Timestamp(END)
    ts = ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')
    end_ts = ts + pd.Timedelta(days=1)
    where.append(f"state_time < TIMESTAMPTZ '{end_ts.isoformat()}'")
if ASSET_FILTER:
    where.append('underlying IN (' + ','.join(repr(x) for x in ASSET_FILTER) + ')')
where_sql = '' if not where else ' WHERE ' + ' AND '.join(where)
sql = (
    'SELECT ' + ', '.join(qident(c) for c in select_cols)
    + ' FROM time_series_feature_state' + where_sql
    + ' ORDER BY underlying, state_time'
)
df = con.execute(sql).df()
con.close()
df['state_time'] = pd.to_datetime(df['state_time'], utc=True)
df = df.reset_index(drop=True)
if df.empty:
    raise RuntimeError('No rows found.')
dupes = int(df.duplicated(['underlying','state_time']).sum())
if dupes:
    raise RuntimeError(f'Found {dupes} duplicate asset/timestamp rows.')
print(f'rows {len(df):,} | assets {sorted(df.underlying.unique())}')
print(df.state_time.min(), '->', df.state_time.max())


## 3. Chronological split and training-only normalization

The UTC-date split is fixed before fitting. Exact-clock robust location/scale parameters are estimated **only on the training period**, with each slot shrunk toward its asset-wide training distribution.

This removes much of the deterministic asset/clock scale without allowing test observations to influence the transformation.


In [ ]:
split_time = vh.chronological_split_time(df, train_fraction=TRAIN_FRACTION)
normalizer = vh.fit_clock_normalizer(df, enhanced_features, split_time)
zdf = vh.apply_clock_normalizer(df, normalizer, enhanced_features)
print('chronological split:', split_time)
print('train rows:', int((zdf.state_time < split_time).sum()), '| test rows:', int((zdf.state_time >= split_time).sum()))
display(normalizer.source.value_counts().rename_axis('normalizer_source').to_frame('n'))


## 4. Training-only q99 target

For each asset, q99 is estimated from the **training-period** distribution of native five-minute variance:

\[
c_i=Q_{0.99}(r_{i,t}^2\mid t<split).
\]

The frozen threshold is then applied to the test chronology. This target remains completely external to HMM fitting.


In [ ]:
q99_thresholds = vh.training_q99_thresholds(df, split_time)
q99_labels = vh.label_q99_onsets_from_train_thresholds(df, q99_thresholds)
display(q99_thresholds)
q99_counts = (
    q99_labels.assign(split=np.where(q99_labels.state_time < split_time, 'train', 'test'))
    .groupby(['split','underlying'])
    .agg(
        q99_bars=('is_q99_train_threshold','sum'),
        q99_onsets=('is_q99_onset_train_threshold','sum'),
        rows=('state_time','size'),
    ).reset_index()
)
display(q99_counts)


## 5. Build exact contiguous core sequences

A sequence ends only when the asset changes, a required core emission is missing, or the timestamp gap is not exactly five minutes. UTC midnight does not reset a crypto state.


In [ ]:
core_train, core_train_idx = vh.build_sequences(zdf, core_features, split_time, split='train')
core_test, core_test_idx = vh.build_sequences(zdf, core_features, split_time, split='test')
core_seq_meta = pd.concat([
    vh.sequence_metadata(zdf, core_train_idx, 'train'),
    vh.sequence_metadata(zdf, core_test_idx, 'test'),
], ignore_index=True)
print('core train sequences:', len(core_train), '| rows:', sum(map(len, core_train)))
print('core test sequences:', len(core_test), '| rows:', sum(map(len, core_test)))
display(core_seq_meta.groupby('split').n_rows.describe())


## 6. Fit baseline HMM grid

For each candidate (K\in\{2,3,4,5\}), several random initializations are fit with Baum–Welch EM using diagonal Gaussian emissions.

`K` is selected by **training BIC subject to non-degenerate occupancy**. Held-out likelihood is reported but does not choose the state count used for q99 evaluation.


In [ ]:
core_compare, core_models = vh.fit_hmm_grid(
    core_train, core_test, CORE_K,
    n_init=N_INIT_CORE,
    max_iter=MAX_ITER_CORE,
    seed_base=6060,
)
core_selected_k = vh.choose_k_by_training_bic(core_compare, min_occupancy=0.01)
core_model = core_models[core_selected_k]
print('selected core K by training BIC:', core_selected_k)
display(core_compare.sort_values('K').round(5))


## 7. Characterize the discovered core states

The state numbers are arbitrary. Economic labels should be assigned only after inspecting their normalized emission profiles and transition matrix.


In [ ]:
core_emissions = vh.state_emission_table(core_model, core_features)
core_transitions = vh.transition_table(core_model)
core_pi = pd.DataFrame({'state': range(core_model.K), 'initial_probability': core_model.pi})
display(core_emissions.pivot(index='state', columns='feature', values='emission_mean_robust_z').round(3))
display(core_transitions.pivot(index='from_state', columns='to_state', values='probability').round(3))


## 8. Held-out filtered state probabilities

Every q99 hazard calculation below uses only **filtered** probabilities:

\[
P(S_t=k\mid X_{1:t}),
\]

never smoothed probabilities that include observations after (t).


In [ ]:
core_filtered_test = vh.filtered_state_table(
    zdf, core_model, core_test, core_test_idx, split='test'
)
display(core_filtered_test[['max_filtered_prob','filtered_entropy']].describe())
display(core_filtered_test.hard_state.value_counts(normalize=True).sort_index().rename('test_hard_occupancy').to_frame())


## 9. External q99 hazard by current state

For each held-out timestamp, define whether a q99 onset occurs later within 30, 60, 90 or 120 minutes. The current timestamp itself is excluded.

The soft-state estimate is:

\[
\hat P(q99_{t+h}\mid S_t=k)
=
\frac{\sum_t P(S_t=k\mid X_{1:t})Y_{t,h}}
{\sum_tP(S_t=k\mid X_{1:t})}.
\]


In [ ]:
core_state_hazard = vh.state_q99_hazard(zdf, core_filtered_test, q99_labels)
display(core_state_hazard.sort_values(['horizon_min','soft_hazard_lift'], ascending=[True,False]).round(4))


## 10. Does a state transition matter more than the state itself?

Hard filtered transitions are evaluated only across exact contiguous five-minute observations. Rare transitions must not be overinterpreted.


In [ ]:
core_transition_hazard = vh.transition_q99_hazard(zdf, core_filtered_test, q99_labels)
display(
    core_transition_hazard[core_transition_hazard.n_transitions >= 20]
    .sort_values(['horizon_min','hazard_lift'], ascending=[True,False])
    .head(80).round(4)
)


## 11. Event-align the filtered states around held-out q99 onsets

This asks whether the −95m motif was the average footprint of a latent state transition. Simultaneous multi-asset q99 events are collapsed by timestamp before the pooled profile.


In [ ]:
core_alignment = vh.event_aligned_state_probabilities(
    zdf, core_filtered_test, q99_labels,
    split_time=split_time,
    rel_bars=range(-24,13),
)
display(core_alignment)
prob_mean_cols = [c for c in core_alignment.columns if c.startswith('p_state_') and c.endswith('_mean')]
if len(core_alignment):
    fig, ax = plt.subplots(figsize=(10,4))
    for c in prob_mean_cols:
        ax.plot(core_alignment.rel_min, core_alignment[c], label=c.replace('_mean',''))
    ax.axvline(-95, linestyle='--', linewidth=1)
    ax.axvline(0, linewidth=1)
    ax.set_xlabel('minutes relative to held-out q99 onset')
    ax.set_ylabel('mean filtered state probability')
    ax.legend(ncol=2)
    plt.show()


## 12. Does the HMM duration assumption look credible?

A standard HMM implies geometric state duration with constant exit probability:

\[
h_k=1-A_{kk}.
\]

We compare this with empirical durations of hard **filtered** test-state runs. Material age dependence or systematic duration mismatch would be evidence for testing an HSMM next.


In [ ]:
core_duration_summary, core_duration_hazard = vh.duration_diagnostics(core_model, core_filtered_test)
display(core_duration_summary.round(3))
for k in range(core_model.K):
    g = core_duration_hazard[core_duration_hazard.state == k]
    if not len(g):
        continue
    fig, ax = plt.subplots(figsize=(7,3))
    ax.plot(g.age_min, g.empirical_exit_hazard, marker='o', label='empirical filtered-run exit hazard')
    ax.axhline(float(g.hmm_constant_exit_hazard.iloc[0]), linewidth=1, label='HMM constant hazard')
    ax.set_title(f'Core state {k}')
    ax.set_xlabel('state age (minutes)')
    ax.set_ylabel('exit hazard')
    ax.legend()
    plt.show()


## 13. Separate sparse-L2 enhancement

The L2 variables are central to the −95m motif but cover much less history. They are therefore added only in a separate model using complete cases for the combined core+L2 feature vector.

This model is exploratory and should not replace the broad-history core HMM unless its states are stable and informative.


In [ ]:
l2_compare = pd.DataFrame()
l2_emissions = pd.DataFrame()
l2_transitions = pd.DataFrame()
l2_pi = pd.DataFrame()
l2_filtered_test = pd.DataFrame()
l2_state_hazard = pd.DataFrame()
l2_transition_hazard = pd.DataFrame()
l2_alignment = pd.DataFrame()
l2_duration_summary = pd.DataFrame()
l2_duration_hazard = pd.DataFrame()
l2_seq_meta = pd.DataFrame()
l2_selected_k = None

if RUN_L2 and len(l2_features) == len(vh.L2_FEATURES):
    l2_train, l2_train_idx = vh.build_sequences(zdf, enhanced_features, split_time, split='train')
    l2_test, l2_test_idx = vh.build_sequences(zdf, enhanced_features, split_time, split='test')
    l2_seq_meta = pd.concat([
        vh.sequence_metadata(zdf, l2_train_idx, 'train'),
        vh.sequence_metadata(zdf, l2_test_idx, 'test'),
    ], ignore_index=True)
    print('L2-enhanced train rows:', sum(map(len,l2_train)), '| test rows:', sum(map(len,l2_test)))
    if len(l2_train) and len(l2_test):
        l2_compare, l2_models = vh.fit_hmm_grid(
            l2_train, l2_test, L2_K,
            n_init=N_INIT_L2,
            max_iter=MAX_ITER_L2,
            seed_base=6600,
        )
        l2_selected_k = vh.choose_k_by_training_bic(l2_compare, min_occupancy=0.01)
        l2_model = l2_models[l2_selected_k]
        print('selected L2-enhanced K by training BIC:', l2_selected_k)
        display(l2_compare.round(5))
        l2_emissions = vh.state_emission_table(l2_model, enhanced_features)
        l2_transitions = vh.transition_table(l2_model)
        l2_pi = pd.DataFrame({'state':range(l2_model.K),'initial_probability':l2_model.pi})
        l2_filtered_test = vh.filtered_state_table(zdf, l2_model, l2_test, l2_test_idx, split='test')
        l2_state_hazard = vh.state_q99_hazard(zdf, l2_filtered_test, q99_labels)
        l2_transition_hazard = vh.transition_q99_hazard(zdf, l2_filtered_test, q99_labels)
        l2_alignment = vh.event_aligned_state_probabilities(zdf, l2_filtered_test, q99_labels, split_time=split_time, rel_bars=range(-24,13))
        l2_duration_summary, l2_duration_hazard = vh.duration_diagnostics(l2_model, l2_filtered_test)
        display(l2_emissions.pivot(index='state', columns='feature', values='emission_mean_robust_z').round(3))
        display(l2_state_hazard.sort_values(['horizon_min','soft_hazard_lift'], ascending=[True,False]).round(4))


## 14. Persist compact outputs


In [ ]:
contract.to_csv(OUT / '06_feature_contract.csv', index=False)
normalizer.to_csv(OUT / '06_clock_normalizer_params.csv', index=False)
q99_thresholds.to_csv(OUT / '06_q99_train_thresholds.csv', index=False)
q99_counts.to_csv(OUT / '06_q99_counts.csv', index=False)
core_seq_meta.to_csv(OUT / '06_core_sequence_metadata.csv', index=False)
core_compare.to_csv(OUT / '06_core_model_comparison.csv', index=False)
core_pi.to_csv(OUT / '06_core_initial_probabilities.csv', index=False)
core_emissions.to_csv(OUT / '06_core_emissions.csv', index=False)
core_transitions.to_csv(OUT / '06_core_transitions.csv', index=False)
core_filtered_test.to_csv(OUT / '06_core_filtered_test.csv', index=False)
core_state_hazard.to_csv(OUT / '06_core_state_q99_hazard.csv', index=False)
core_transition_hazard.to_csv(OUT / '06_core_transition_q99_hazard.csv', index=False)
core_alignment.to_csv(OUT / '06_core_q99_alignment.csv', index=False)
core_duration_summary.to_csv(OUT / '06_core_duration_summary.csv', index=False)
core_duration_hazard.to_csv(OUT / '06_core_duration_hazard.csv', index=False)

if len(l2_compare):
    l2_seq_meta.to_csv(OUT / '06_l2_sequence_metadata.csv', index=False)
    l2_compare.to_csv(OUT / '06_l2_model_comparison.csv', index=False)
    l2_pi.to_csv(OUT / '06_l2_initial_probabilities.csv', index=False)
    l2_emissions.to_csv(OUT / '06_l2_emissions.csv', index=False)
    l2_transitions.to_csv(OUT / '06_l2_transitions.csv', index=False)
    l2_filtered_test.to_csv(OUT / '06_l2_filtered_test.csv', index=False)
    l2_state_hazard.to_csv(OUT / '06_l2_state_q99_hazard.csv', index=False)
    l2_transition_hazard.to_csv(OUT / '06_l2_transition_q99_hazard.csv', index=False)
    l2_alignment.to_csv(OUT / '06_l2_q99_alignment.csv', index=False)
    l2_duration_summary.to_csv(OUT / '06_l2_duration_summary.csv', index=False)
    l2_duration_hazard.to_csv(OUT / '06_l2_duration_hazard.csv', index=False)

meta = {
    'version': vh.VERSION,
    'module': '06_latent_market_state_hmm',
    'model_class': 'first-order diagonal-Gaussian HMM',
    'numpy_only_implementation': True,
    'q99_used_in_fit': False,
    'return_5m_used_as_emission': False,
    'state_labels_forced': False,
    'duration_model': 'standard HMM geometric implication only',
    'train_fraction_by_utc_date': TRAIN_FRACTION,
    'split_time': str(split_time),
    'normalization': 'training-only asset/exact-clock robust z with shrinkage',
    'core_features': core_features,
    'l2_features': l2_features,
    'core_candidate_K': list(CORE_K),
    'core_selected_K_training_BIC': int(core_selected_k),
    'l2_candidate_K': list(L2_K),
    'l2_selected_K_training_BIC': int(l2_selected_k) if l2_selected_k is not None else None,
    'future_q99_horizons_min': list(vh.DEFAULT_FUTURE_HORIZONS_MIN),
    'heldout_hazard_uses_filtered_states_only': True,
    'smoothing_used_for_heldout_decisions': False,
    'hsmm_not_fit': True,
    'start': START or 'earliest_available',
    'end': END or 'latest_available',
    'n_rows': int(len(df)),
}
(OUT / '06_run_metadata.json').write_text(json.dumps(meta, indent=2))
print('wrote Module 06 outputs to', OUT)


## Interpretation boundary

Module 06 can establish that recurrent hidden configurations exist, characterize them, and test whether their **held-out filtered probabilities** are associated with future q99 hazard.

It does not yet establish an HSMM, a Bridge state, or a trading signal. An HSMM is justified only if state-age/duration diagnostics show that the HMM's constant exit hazard is materially inadequate and the richer duration model subsequently improves chronological out-of-sample performance.
